In [ ]:
# Instalo librerías para correr en colab o local
# Si es en local, se debe instalar la versión de torch. Only CPU o con GPU (fijarse la versión de CUDA)
!pip install transformers datasets peft bitsandbytes accelerate numpy sentencepiece huggingface_hub notebook>=7 ipywidgets wandb

In [18]:
!hf auth login --force

^C


In [2]:
# Import librerías necesarias
import random
from datetime import datetime
from dataclasses import dataclass, field
from typing import Dict, List
import numpy as np
import torch
import wandb
from datasets import Dataset
from transformers import(
    AutoModelForCausalLM, #
    AutoTokenizer,        #
    BitsAndBytesConfig,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
)
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
)
from dataset_generator import generate_training_dataset

W1016 00:05:53.423000 26580 Lib\site-packages\torch\utils\flop_counter.py:113] triton not found; flop counting will not work for triton kernels


In [3]:
@dataclass
class Config:
  # Modelo
  model_name: str = 'google/gemma-2b'
  output_dir: str = './gemma_lora_json_adapter'

  # LoRa
  lora_r: int = 16 # Lado de la matriz. D x r. D lado de la matriz del modelo.
  lora_alpha: int = 32 # Factor que afecta la influencia del LoRa sobre el modelo. (a/b)r * B.A
  # B : matriz de ceros
  # A : matriz de números aleatorios
  lora_dropout: float = 0.05 # para evitar overfitting
  # https://developers.googleblog.com/gemma-explained-overview-gemma-model-family-architectures/
  lora_target_modules: List[str] = field(
      default_factory=lambda: ['q_proj', 'k_proj', 'v_proj', 'o_proj']
      ) # Capas que estoy refinando con el LoRa
  # Quantization (QLoRa)
  use_4bit: bool = True
  bnb_4bit_compute_dtype: str = 'bfloat16'
  bnb_4bit_quant_type: str = 'nf4'
  use_nested_quant: bool = True

  # Training
  num_train_epochs: int = 3
  max_steps: int = 300
  per_device_train_batch_size: int = 2
  per_device_eval_batch_size: int = 2
  gradient_accumulation_steps: int = 8 # Evita actualiza en cada steps, permite simular batch más grandes
  learning_rate: float = 2e-4
  weight_decay: float = 0.01
  warmup_ratio: float = 0.03
  max_seq_length: int = 256

  # Generation parameters
  max_new_tokens: int = 120
  temperature: float = 1.0
  top_p: float = 1.0
  do_sample: bool = False
  # do_sample False, ayuda a generar LoRa más determinísticos. ignora el temperature y top_p.

  # Dataset
  train_size: int = 400
  eval_size: int = 100

  # Reproducibility
  seed: int = 42


LOW_VRAM_OVERRIDES = {
    'lora_r': 8,
    'lora_alpha': 16,
    'max_sep_lenght': 128,
    'max_steps': 100,
    'output_dir': './gemma_lora_json_adapter_lowvram'
}

config = Config()

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)

set_seed(config.seed)

In [4]:
def load_model_and_tokenizer(prepare_for_training: bool = True):
    print(f'Modelo cargado: {config.model_name}')
    
    bnb_config = BitsAndBytesConfig(  # Configuro la quantización.
        load_in_4bit=config.use_4bit,
        bnb_4bit_quant_type=config.bnb_4bit_quant_type,
        bnb_4bit_compute_dtype=getattr(torch, config.bnb_4bit_compute_dtype),
        bnb_4bit_use_double_quant=config.use_nested_quant,
    )
    tokenizer = AutoTokenizer.from_pretrained(config.model_name, trust_remote_code = True)

    if tokenizer.pad_token is None: # En caso no haya 
        tokenizer.pad_token = tokenizer.eos_token
        tokenizer.padding_size = 'right'

    model = AutoModelForCausalLM.from_pretrained(
        config.model_name,
        quantization_config = bnb_config,
        device_map = 'auto',
        trust_remote_code = True,
    )

    if prepare_for_training:
        model = prepare_model_for_kbit_training(model)

    print(f'Loaded - {model.get_memory_footprint() / 1e9:.2f} GB\n')
    return model, tokenizer


In [5]:
model, tokenizer = load_model_and_tokenizer()

Modelo cargado: google/gemma-2b


[transformers] We found `hidden_act="gelu"` in this Gemma config. This is a legacy value of the official releases but it is meant to target the tanh approximation. Setting `hidden_act="gelu_pytorch_tanh"` instead.


Loading weights:   0%|          | 0/164 [00:00<?, ?it/s]

Loaded - 3.09 GB



In [6]:
# Lora
def apply_lora(model):
    lora_cfg = LoraConfig(
        r = config.lora_r,
        lora_alpha = config.lora_alpha,
        target_modules = config.lora_target_modules,
        lora_dropout=config.lora_dropout,
        bias = 'none',
        task_type='CAUSAL_LM',
    )

    model = get_peft_model(model, lora_cfg)

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print(f'Lora Applied - trainable: {trainable:,} / {total:,}'
          f'({100 * trainable / total:.2f}%)\n')
    return model

lora_temp = apply_lora(model)

# Nota, por la quantización hace parecer que hay menos parámetros, pero sigue habiendo 2.5


Lora Applied - trainable: 3,686,400 / 1,518,954,496(0.24%)



In [7]:
def format_prompt(example:Dict) -> str: # Formato de los prompt de entrenamiento
    return(
        f'<start_of_turn>user\n'
        f'{example['instruction']}\n\n'
        f'Customer message: {example['input']}<end_of_turn>\n'
        f'<start_of_turn>model\n'
        f'{example['output']}<end_of_turn>'
        )

def tokenize_examples(examples: Dict, tokenizer) -> Dict:
    prompts = [
        format_prompt({'instruction': i, 'input': inp, 'output':o})
        for i, inp, o in zip(
            examples['instruction'], examples['input'],examples['output']
        )
    ]

    tok = tokenizer(
            prompts,
            truncation=True,
            max_length=config.max_seq_length,
            padding="max_length",
            return_tensors="pt",
        )
    tok["labels"] = tok["input_ids"].clone()
    return tok


In [8]:
def train(model, tokenizer, train_ds, eval_ds):
    args = TrainingArguments(
        output_dir = config.output_dir,
        num_train_epochs=config.num_train_epochs,
        max_steps=config.max_steps if config.max_steps > 0 else -1,
        per_device_train_batch_size=config.per_device_train_batch_size,
        per_device_eval_batch_size=config.per_device_eval_batch_size,
        gradient_accumulation_steps=config.gradient_accumulation_steps,
        learning_rate=config.learning_rate,
        weight_decay=config.weight_decay,
        warmup_steps=config.warmup_ratio,
        logging_steps=10,
        eval_strategy='steps' if config.max_steps > 0 else 'epoch',
        eval_steps = 50  if config.max_steps > 0  else None,
        save_strategy= 'steps' if config.max_steps > 0 else 'epoch',
        save_steps = 100 if config.max_steps>0 else None,
        save_total_limit=2,
        # Load_best_model_at_ends = True
        # metric_for_best_model='eval_loss'
        fp16 = False,
        bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported(),
        optim='paged_adamw_8bit',
        lr_scheduler_type='cosine',
        seed=config.seed,       
    )

    trainer = Trainer(
        model = model,
        args = args,
        train_dataset=train_ds,
        eval_dataset=eval_ds,
        data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm = False), # Para modelos autoregresivos, modelos tipo BERT (masked prediction) colocar True

    )

    eff_bs = config.per_device_train_batch_size * config.gradient_accumulation_steps 
    print(f'Training - lr={config.learning_rate}, eff_batch={eff_bs}, max_steps = {config.max_steps}\n')

    trainer.train()
    model.save_pretrained(config.output_dir)
    tokenizer.save_pretrained(config.output_dir)
    print(f'Adapter saved to  {config.output_dir}')

    return trainer


In [9]:
# Main pipeline

print(f"Model: {config.model_name}  |  LoRA r={config.lora_r}  |  "
          f"steps={config.max_steps}  |  seed={config.seed}\n")

# 1. Dataset
print("[1/4] Generating dataset")
train_ex, eval_ex = generate_training_dataset(
        train_size=config.train_size, eval_size=config.eval_size,
    )
print(f"  train={len(train_ex)}  eval={len(eval_ex)}\n")

# 2. Load model & apply LoRA 
print("[2/4] Loading model + LoRA")
model, tokenizer = load_model_and_tokenizer(prepare_for_training=True)
model = apply_lora(model)

# 3. Tokenize
print("[3/4] Tokenising")
train_ds = Dataset.from_list(train_ex).map(
        lambda x: tokenize_examples(x, tokenizer),
        batched=True, remove_columns=["instruction", "input", "output"],
    )
eval_ds = Dataset.from_list(eval_ex).map(
        lambda x: tokenize_examples(x, tokenizer),
        batched=True, remove_columns=["instruction", "input", "output"],
    )
print("done\n")

# 4. Train 
print("[4/4] Training")
train(model, tokenizer, train_ds, eval_ds)

print("TRAINING COMPLETE")
print(f"  Adapter saved to: {config.output_dir}")

Model: google/gemma-2b  |  LoRA r=16  |  steps=300  |  seed=42

[1/4] Generating dataset
  train=400  eval=100

[2/4] Loading model + LoRA
Modelo cargado: google/gemma-2b


Loading weights:   0%|          | 0/164 [00:00<?, ?it/s]

Loaded - 3.09 GB

Lora Applied - trainable: 3,686,400 / 1,518,954,496(0.24%)

[3/4] Tokenising


Map:   0%|          | 0/400 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

done

[4/4] Training
Training - lr=0.0002, eff_batch=16, max_steps = 300



c:\Users\rober\Desktop\utec_lora\venv-lora\Lib\site-packages\torch\_dynamo\eval_frame.py:1548: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss,Validation Loss
50,0.613410,0.568143
100,0.473284,0.466473
150,0.437619,0.436439
200,0.409070,0.405602
250,0.392658,0.391769
300,0.390281,0.389230


c:\Users\rober\Desktop\utec_lora\venv-lora\Lib\site-packages\torch\_dynamo\eval_frame.py:1548: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
c:\Users\rober\Desktop\utec_lora\venv-lora\Lib\site-packages\torch\_dynamo\eval_frame.py:1548: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  ret

Adapter saved to  ./gemma_lora_json_adapter
TRAINING COMPLETE
  Adapter saved to: ./gemma_lora_json_adapter
